In [7]:
"""
DEPLOYMENT CELL 1: SETUP AND LOAD TRAINED MODELS
Load the trained two-stage XGBoost models for prediction
"""

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import requests
import joblib
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("24-HOUR RAINFALL PREDICTION - DEPLOYMENT")
print("="*70)
print(f"Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

# Configuration
COLOMBO_LAT = 6.9271
COLOMBO_LON = 79.8612
CLASSIFICATION_THRESHOLD = 1.0  # mm

print("Loading trained models...")

# Load Stage 1 (Classification)
try:
    stage1_model = joblib.load('data/stage1_rain_classifier.pkl')
    print("   ✓ Stage 1 (Rain Classifier) loaded")
except FileNotFoundError:
    print("   ✗ Error: data/stage1_rain_classifier.pkl not found")
    print("   → Make sure you're running this in the same directory as training notebook")
    raise

# Load Stage 2 (Regression)
try:
    stage2_model = joblib.load('data/stage2_amount_regressor.pkl')
    print("   ✓ Stage 2 (Amount Regressor) loaded")
except FileNotFoundError:
    print("   ✗ Error: data/stage2_amount_regressor.pkl not found")
    raise

# Load feature names (to ensure correct order)
try:
    with open('data/feature_names.txt', 'r', encoding='utf-8') as f:
        feature_names = [line.strip() for line in f.readlines()]
    print(f"   ✓ Feature names loaded ({len(feature_names)} features)")
except FileNotFoundError:
    print("Warning: feature_names.txt not found")
    print("   → Will use model's feature names (may have ordering issues)")
    feature_names = None

print(f"\n✓ Models loaded successfully")
print(f"\nConfiguration:")
print(f"  Location: Colombo, Sri Lanka ({COLOMBO_LAT}°N, {COLOMBO_LON}°E)")
print(f"  Classification threshold: {CLASSIFICATION_THRESHOLD} mm")


24-HOUR RAINFALL PREDICTION - DEPLOYMENT
Current time: 2026-03-07 11:04:42

Loading trained models...
   ✓ Stage 1 (Rain Classifier) loaded
   ✓ Stage 2 (Amount Regressor) loaded
   ✓ Feature names loaded (71 features)

✓ Models loaded successfully

Configuration:
  Location: Colombo, Sri Lanka (6.9271°N, 79.8612°E)
  Classification threshold: 1.0 mm


In [8]:
"""
DEPLOYMENT CELL 2: FETCH CURRENT AND HISTORICAL DATA
Get recent weather data from Open-Meteo API
Needs 20 days of history for daily lag features (max lag = 14 days)
"""

print("\n" + "="*70)
print("FETCHING WEATHER DATA FROM OPEN-METEO API")
print("="*70)

def fetch_weather_data_for_prediction():
    """
    Fetch weather data for making a prediction RIGHT NOW.
    Strategy:
      - Archive API: 20 days ago to 3 days ago (stable historical data)
      - Forecast API: past_days=4 (recent + current observations)
    Combined gives ~20 days of hourly data — enough for all lag features.
    """

    current_time = datetime.now()
    current_date = current_time.date()

    print(f"\nFetching weather data...")
    print(f"   Current time: {current_time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"   Location: {COLOMBO_LAT}N, {COLOMBO_LON}E")

    hourly_vars = [
        "temperature_2m",
        "relative_humidity_2m",
        "dew_point_2m",
        "precipitation",
        "surface_pressure",
        "cloud_cover",
        "wind_speed_10m",
        "wind_direction_10m"
    ]

    # ── PART 1: Archive API (stable historical data) ──────────────────────
    print(f"\n   1. Fetching historical data (Archive API)...")

    archive_start = current_date - timedelta(days=20)
    archive_end   = current_date - timedelta(days=4)

    archive_params = {
        "latitude":   COLOMBO_LAT,
        "longitude":  COLOMBO_LON,
        "start_date": str(archive_start),
        "end_date":   str(archive_end),
        "hourly":     ",".join(hourly_vars),
        "timezone":   "Asia/Colombo"
    }

    df_historical = None
    try:
        response = requests.get(
            "https://archive-api.open-meteo.com/v1/archive",
            params=archive_params, timeout=30
        )
        response.raise_for_status()
        data = response.json()['hourly']

        df_historical = pd.DataFrame({
            'datetime':             pd.to_datetime(data['time']),
            'temperature_2m':       data['temperature_2m'],
            'relative_humidity_2m': data['relative_humidity_2m'],
            'dew_point_2m':         data['dew_point_2m'],
            'precipitation':        data['precipitation'],
            'surface_pressure':     data['surface_pressure'],
            'cloud_cover':          data['cloud_cover'],
            'wind_speed_10m':       data['wind_speed_10m'],
            'wind_direction_10m':   data['wind_direction_10m']
        })
        print(f"      Fetched {len(df_historical)} historical records")
        print(f"      Range: {df_historical['datetime'].min()} to {df_historical['datetime'].max()}")

    except Exception as e:
        print(f"      Archive API error: {e}")

    # ── PART 2: Forecast API (recent + current observations) ─────────────
    print(f"\n   2. Fetching recent/current data (Forecast API)...")

    forecast_params = {
        "latitude":      COLOMBO_LAT,
        "longitude":     COLOMBO_LON,
        "hourly":        ",".join(hourly_vars),
        "past_days":     4,
        "forecast_days": 1,
        "timezone":      "Asia/Colombo"
    }

    df_recent = None
    try:
        response = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params=forecast_params, timeout=30
        )
        response.raise_for_status()
        data = response.json()['hourly']

        df_recent = pd.DataFrame({
            'datetime':             pd.to_datetime(data['time']),
            'temperature_2m':       data['temperature_2m'],
            'relative_humidity_2m': data['relative_humidity_2m'],
            'dew_point_2m':         data['dew_point_2m'],
            'precipitation':        data['precipitation'],
            'surface_pressure':     data['surface_pressure'],
            'cloud_cover':          data['cloud_cover'],
            'wind_speed_10m':       data['wind_speed_10m'],
            'wind_direction_10m':   data['wind_direction_10m']
        })

        # Keep only past/current observations — exclude future forecast hours
        df_recent = df_recent[df_recent['datetime'] <= current_time].copy()
        print(f"      Fetched {len(df_recent)} recent records")
        print(f"      Latest: {df_recent['datetime'].max()}")

    except Exception as e:
        print(f"      Forecast API error: {e}")

    # ── PART 3: Combine ───────────────────────────────────────────────────
    print(f"\n   3. Combining datasets...")

    dfs = [d for d in [df_historical, df_recent] if d is not None]

    if len(dfs) == 0:
        print(f"      Failed to fetch any data")
        return None

    df_combined = pd.concat(dfs, ignore_index=True)
    df_combined = df_combined.drop_duplicates(subset=['datetime'], keep='last')
    df_combined = df_combined.sort_values('datetime').reset_index(drop=True)

    print(f"      Combined: {len(df_combined)} records")
    print(f"      Range: {df_combined['datetime'].min()} to {df_combined['datetime'].max()}")

    hours_behind = (current_time - df_combined['datetime'].max()).total_seconds() / 3600
    if hours_behind > 3:
        print(f"      Warning: Latest data is {hours_behind:.1f}h behind current time")
        print(f"      This is normal for Open-Meteo free tier")
    else:
        print(f"      Data is up to date ({hours_behind:.1f}h lag)")

    # Verify enough history for 14-day lags
    days_available = (df_combined['datetime'].max() - df_combined['datetime'].min()).days
    if days_available < 15:
        print(f"      Warning: Only {days_available} days of history available")
        print(f"      Lag features beyond {days_available}d will be NaN")
    else:
        print(f"      {days_available} days of history — sufficient for all lag features")

    return df_combined

# Fetch data
df_raw = fetch_weather_data_for_prediction()

if df_raw is None:
    raise Exception("Failed to fetch weather data. Cannot make prediction.")

print(f"\nWeather data ready:")
print(f"   Records:          {len(df_raw):,}")
print(f"   Date range:       {df_raw['datetime'].min().date()} to {df_raw['datetime'].max().date()}")
print(f"   Latest timestamp: {df_raw['datetime'].max()}")



FETCHING WEATHER DATA FROM OPEN-METEO API

Fetching weather data...
   Current time: 2026-03-07 11:04:42
   Location: 6.9271N, 79.8612E

   1. Fetching historical data (Archive API)...
      Fetched 408 historical records
      Range: 2026-02-15 00:00:00 to 2026-03-03 23:00:00

   2. Fetching recent/current data (Forecast API)...
      Fetched 108 recent records
      Latest: 2026-03-07 11:00:00

   3. Combining datasets...
      Combined: 492 records
      Range: 2026-02-15 00:00:00 to 2026-03-07 11:00:00
      Data is up to date (0.1h lag)
      20 days of history — sufficient for all lag features

Weather data ready:
   Records:          492
   Date range:       2026-02-15 to 2026-03-07
   Latest timestamp: 2026-03-07 11:00:00


In [9]:
"""
DEPLOYMENT CELL 3: FEATURE ENGINEERING PIPELINE
Apply exact same transformations as training.
Computes ONLY the 71 clean leakage-free features.
Removed from old code: daily aggregates, cumsum, consecutive,
rolling means, historical stats — all were leakers in training.
"""

print("\n" + "="*70)
print("FEATURE ENGINEERING (71 CLEAN FEATURES)")
print("="*70)

def engineer_features_for_prediction(df):
    """
    Replicates training feature engineering exactly.
    Returns the LAST ROW with all 71 clean features populated.
    """

    print(f"\nEngineering features...")
    df = df.sort_values('datetime').reset_index(drop=True)

    # ── Datetime components ───────────────────────────────────────────────
    df['hour']        = df['datetime'].dt.hour
    df['month']       = df['datetime'].dt.month
    df['year']        = df['datetime'].dt.year
    df['day_of_year'] = df['datetime'].dt.dayofyear
    df['day_of_week'] = df['datetime'].dt.dayofweek
    df['date_dt']     = df['datetime'].dt.normalize()

    # ── Rolling 24h features (28) ─────────────────────────────────────────
    W = 24
    MP = 18  # min_periods

    df['precip_24h_sum']   = df['precipitation'].rolling(W, min_periods=MP).sum()
    df['precip_24h_max']   = df['precipitation'].rolling(W, min_periods=MP).max()
    df['precip_24h_mean']  = df['precipitation'].rolling(W, min_periods=MP).mean()
    df['precip_24h_std']   = df['precipitation'].rolling(W, min_periods=MP).std()
    df['precip_24h_rainy_hours'] = df['precipitation'].rolling(W, min_periods=MP).apply(
        lambda x: (x > 0.1).sum(), raw=True
    )

    df['temp_24h_mean']  = df['temperature_2m'].rolling(W, min_periods=MP).mean()
    df['temp_24h_max']   = df['temperature_2m'].rolling(W, min_periods=MP).max()
    df['temp_24h_min']   = df['temperature_2m'].rolling(W, min_periods=MP).min()
    df['temp_24h_range'] = df['temp_24h_max'] - df['temp_24h_min']
    df['temp_24h_std']   = df['temperature_2m'].rolling(W, min_periods=MP).std()
    df['temp_24h_trend'] = df['temperature_2m'] - df['temperature_2m'].shift(W)

    df['humidity_24h_mean']  = df['relative_humidity_2m'].rolling(W, min_periods=MP).mean()
    df['humidity_24h_max']   = df['relative_humidity_2m'].rolling(W, min_periods=MP).max()
    df['humidity_24h_min']   = df['relative_humidity_2m'].rolling(W, min_periods=MP).min()
    df['humidity_24h_range'] = df['humidity_24h_max'] - df['humidity_24h_min']
    df['humidity_24h_hours_above_80'] = df['relative_humidity_2m'].rolling(W, min_periods=MP).apply(
        lambda x: (x > 80).sum(), raw=True
    )

    df['pressure_24h_mean']  = df['surface_pressure'].rolling(W, min_periods=MP).mean()
    df['pressure_24h_min']   = df['surface_pressure'].rolling(W, min_periods=MP).min()
    df['pressure_24h_max']   = df['surface_pressure'].rolling(W, min_periods=MP).max()
    df['pressure_24h_std']   = df['surface_pressure'].rolling(W, min_periods=MP).std()
    df['pressure_24h_trend'] = df['surface_pressure'] - df['surface_pressure'].shift(W)

    df['wind_24h_mean'] = df['wind_speed_10m'].rolling(W, min_periods=MP).mean()
    df['wind_24h_max']  = df['wind_speed_10m'].rolling(W, min_periods=MP).max()
    df['wind_24h_std']  = df['wind_speed_10m'].rolling(W, min_periods=MP).std()

    df['cloud_24h_mean'] = df['cloud_cover'].rolling(W, min_periods=MP).mean()
    df['cloud_24h_max']  = df['cloud_cover'].rolling(W, min_periods=MP).max()
    df['cloud_24h_hours_above_90'] = df['cloud_cover'].rolling(W, min_periods=MP).apply(
        lambda x: (x > 90).sum(), raw=True
    )

    df['dewpoint_24h_mean'] = df['dew_point_2m'].rolling(W, min_periods=MP).mean()

    print(f"   1. Rolling 24h features computed (28)")

    # ── Current state features (10) ───────────────────────────────────────
    df['temp_current']              = df['temperature_2m']
    df['humidity_current']          = df['relative_humidity_2m']
    df['pressure_current']          = df['surface_pressure']
    df['dewpoint_current']          = df['dew_point_2m']
    df['wind_speed_current']        = df['wind_speed_10m']
    df['wind_direction_current']    = df['wind_direction_10m']
    df['cloud_cover_current']       = df['cloud_cover']
    df['dew_point_depression_current'] = df['temperature_2m'] - df['dew_point_2m']
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

    print(f"   2. Current state features computed (10)")

    # ── Daily aggregates for lag computation only ─────────────────────────
    daily_agg = df.groupby('date_dt').agg(
        daily_rainfall_total=('precipitation', 'sum'),
        daily_temp_mean=('temperature_2m', 'mean'),
        daily_pressure_mean=('surface_pressure', 'mean'),
        daily_humidity_mean=('relative_humidity_2m', 'mean'),
    ).reset_index().sort_values('date_dt').reset_index(drop=True)

    for lag in [1, 2, 3, 7, 14]:
        daily_agg[f'rainfall_lag_{lag}d'] = daily_agg['daily_rainfall_total'].shift(lag)
    for lag in [1, 3, 7]:
        daily_agg[f'temp_lag_{lag}d'] = daily_agg['daily_temp_mean'].shift(lag)
    for lag in [1, 3]:
        daily_agg[f'pressure_lag_{lag}d'] = daily_agg['daily_pressure_mean'].shift(lag)
    for lag in [1, 3]:
        daily_agg[f'humidity_lag_{lag}d'] = daily_agg['daily_humidity_mean'].shift(lag)

    lag_cols = [c for c in daily_agg.columns if '_lag_' in c]
    df = df.merge(daily_agg[['date_dt'] + lag_cols], on='date_dt', how='left')

    print(f"   3. Daily lag features computed (12)")

    # ── Temporal / seasonal features (14) ────────────────────────────────
    df['month_sin']        = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos']        = np.cos(2 * np.pi * df['month'] / 12)
    df['day_of_year_sin']  = np.sin(2 * np.pi * df['day_of_year'] / 365.25)
    df['day_of_year_cos']  = np.cos(2 * np.pi * df['day_of_year'] / 365.25)

    df['season_SW_Monsoon']  = df['month'].isin([5, 6, 7]).astype(int)
    df['season_NE_Monsoon']  = df['month'].isin([12, 1, 2]).astype(int)
    df['season_Inter_Heavy'] = df['month'].isin([4, 10, 11]).astype(int)
    df['season_Inter_Light'] = (~df['month'].isin([5,6,7,12,1,2,4,10,11])).astype(int)

    def days_into_sw(row):
        if row['month'] in [5, 6, 7]:
            return (row['datetime'] - pd.Timestamp(year=row['year'], month=5, day=1)).days
        return 0

    def days_into_ne(row):
        if row['month'] in [12, 1, 2]:
            ref_year = row['year'] if row['month'] == 12 else row['year'] - 1
            return (row['datetime'] - pd.Timestamp(year=ref_year, month=12, day=1)).days
        return 0

    df['days_into_sw_monsoon'] = df.apply(days_into_sw, axis=1)
    df['days_into_ne_monsoon'] = df.apply(days_into_ne, axis=1)

    print(f"   4. Temporal/seasonal features computed (14)")

    # ── Derived atmospheric features (7) ──────────────────────────────────
    df['pressure_tendency_3h'] = (df['surface_pressure'] - df['surface_pressure'].shift(3)) / 3
    df['pressure_tendency_6h'] = (df['surface_pressure'] - df['surface_pressure'].shift(6)) / 6

    df['wind_from_southwest'] = (
        (df['wind_direction_10m'] >= 180) & (df['wind_direction_10m'] <= 270)
    ).astype(int)
    df['wind_from_northeast'] = (
        (df['wind_direction_10m'] >= 0) & (df['wind_direction_10m'] <= 90)
    ).astype(int)

    wind_dir_6h_ago = df['wind_direction_10m'].shift(6)
    raw_change = np.abs(df['wind_direction_10m'] - wind_dir_6h_ago)
    df['wind_direction_change_6h'] = raw_change.apply(
        lambda x: min(x, 360 - x) if pd.notna(x) else x
    )

    df['instability_index'] = (
        df['temp_24h_range'] * (1 - df['dew_point_depression_current'] / 20)
    ).clip(lower=0)

    df['moisture_flux'] = df['humidity_24h_mean'] * df['wind_24h_mean']

    print(f"   5. Derived atmospheric features computed (7)")

    # ── DROP intermediate columns, return only the 71 clean features ──────
    # Removes: datetime, 8 raw API cols, year, date_dt
        # NEW tail — capture prediction_time BEFORE dropping datetime:
    prediction_time = df['datetime'].iloc[-1]          # ← NEW: save before drop

    cols_to_drop = [
        'datetime', 'year', 'date_dt',
        'temperature_2m', 'relative_humidity_2m', 'dew_point_2m',
        'precipitation', 'surface_pressure', 'cloud_cover',
        'wind_speed_10m', 'wind_direction_10m'
    ]
    df = df.drop(columns=cols_to_drop)
    assert df.shape[1] == 71, f"Expected 71 features, got {df.shape[1]}"
    print(f"   6. Dropped 11 intermediate columns → {df.shape[1]} clean features confirmed")
    return df, prediction_time                         # ← CHANGED: return tuple


# Apply feature engineering
df_features, prediction_time = engineer_features_for_prediction(df_raw)
print(f"\nFeature engineering complete: {df_features.shape[1]} columns")  # Still prints 71




FEATURE ENGINEERING (71 CLEAN FEATURES)

Engineering features...
   1. Rolling 24h features computed (28)
   2. Current state features computed (10)
   3. Daily lag features computed (12)
   4. Temporal/seasonal features computed (14)
   5. Derived atmospheric features computed (7)
   6. Dropped 11 intermediate columns → 71 clean features confirmed

Feature engineering complete: 71 columns


In [10]:
# At the top of Cell 4 — load both models' expected feature orders
FEATURE_NAMES_CLF = stage1_model.get_booster().feature_names  # Stage 1 classifier
FEATURE_NAMES_REG = stage2_model.get_booster().feature_names  # Stage 2 regressor

assert FEATURE_NAMES_CLF == FEATURE_NAMES_REG, \
    "Warning: Stage 1 and Stage 2 were trained with different feature sets!"

FEATURE_NAMES = FEATURE_NAMES_CLF  # Use this for X_pred construction


In [11]:
"""
DEPLOYMENT CELL 4: MAKE PREDICTION FOR CURRENT HOUR
Select the 71 clean features, align with training order, predict.
"""

print("\n" + "="*70)
print("MAKING PREDICTION")
print("="*70)

# NEW — exact order the model was trained with, guaranteed correct
FEATURE_NAMES = stage1_model.get_booster().feature_names
print(f"Feature order loaded from model: {len(FEATURE_NAMES)} features")
print(f"First 4: {FEATURE_NAMES[:4]}")   # Will show: hour, month, day_of_year, day_of_week

# Get latest row
latest_row = df_features.iloc[-1:].copy()
# prediction_time = latest_row['datetime'].values[0]

print(f"\nPrediction time:  {pd.to_datetime(prediction_time)}")
print(f"Target window:    Next 24 hours")
print(f"Expected features: {len(FEATURE_NAMES)}")

# Build feature vector in exact training order
X_pred = pd.DataFrame(index=[0], columns=FEATURE_NAMES, dtype=float)

missing_features = []
for feat in FEATURE_NAMES:
    if feat in latest_row.columns:
        X_pred[feat] = latest_row[feat].values[0]
    else:
        X_pred[feat] = 0.0
        missing_features.append(feat)

if missing_features:
    print(f"\nWarning: {len(missing_features)} features missing, defaulted to 0:")
    for f in missing_features:
        print(f"   - {f}")
else:
    print(f"All {len(FEATURE_NAMES)} features present")

# Handle NaN values
nan_count = X_pred.isnull().sum().sum()
if nan_count > 0:
    print(f"\nWarning: {nan_count} NaN values found — filling with 0")
    X_pred = X_pred.fillna(0)

X_pred = X_pred.astype(float)

# Verify model compatibility
try:
    _ = stage1_model.predict_proba(X_pred)
    print(f"Feature set compatible with model")
except Exception as e:
    print(f"Feature compatibility error: {e}")
    raise

# ── Stage 1: Rain probability ──────────────────────────────────────────────
print(f"\nRunning Stage 1 (Classification)...")
rain_probability = stage1_model.predict_proba(X_pred)[0, 1]
rain_prediction  = rain_probability >= 0.5
print(f"   P(rain >= {CLASSIFICATION_THRESHOLD}mm) = {rain_probability:.3f}")
print(f"   Classification: {'RAIN' if rain_prediction else 'NO RAIN'}")

# ── Stage 2: Rainfall amount ───────────────────────────────────────────────
print(f"\nRunning Stage 2 (Regression)...")
predicted_amount = max(0, stage2_model.predict(X_pred)[0])
print(f"   Predicted amount: {predicted_amount:.2f} mm")

# ── Combined prediction ────────────────────────────────────────────────────
final_prediction = max(0, rain_probability * predicted_amount)

print(f"\n{'='*70}")
print(f"FINAL PREDICTION")
print(f"{'='*70}")
print(f"\n   Next 24-hour rainfall: {final_prediction:.2f} mm")
print(f"\n   Breakdown:")
print(f"      Rain probability:    {rain_probability:.1%}")
print(f"      Predicted amount:    {predicted_amount:.2f} mm")
print(f"      Combined (P x amt):  {final_prediction:.2f} mm")

# Data freshness
data_age_hours = (datetime.now() - pd.to_datetime(prediction_time)).total_seconds() / 3600
print(f"\n   Observation time: {pd.to_datetime(prediction_time).strftime('%Y-%m-%d %H:%M')}")
print(f"   Data age:         {data_age_hours:.1f} hours")

# ── Interpretation ────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"INTERPRETATION")
print(f"{'='*70}")

if final_prediction < 1:
    category          = "No significant rain"
    irrigation_advice = "Normal irrigation recommended"
elif final_prediction < 10:
    category          = "Light rain"
    irrigation_advice = "Reduce irrigation by 30-50%"
elif final_prediction < 50:
    category          = "Moderate rain"
    irrigation_advice = "Skip irrigation today"
else:
    category          = "Heavy rain"
    irrigation_advice = "Skip irrigation, check drainage"

season_map = {
    'season_SW_Monsoon':  'SW Monsoon (May-Jul)',
    'season_NE_Monsoon':  'NE Monsoon (Dec-Feb)',
    'season_Inter_Heavy': 'Inter-Monsoon Wet (Apr, Oct, Nov)',
    'season_Inter_Light': 'Dry Season (Mar, Aug, Sep)',
}
current_season = next(
    (name for col, name in season_map.items() if X_pred[col].values[0] == 1),
    "Unknown"
)

print(f"\n   Category:   {category}")
print(f"   For CWR:    {irrigation_advice}")
print(f"   Season:     {current_season}")

# ── Store result ──────────────────────────────────────────────────────────
prediction_result = {
    'prediction_time':      str(pd.to_datetime(prediction_time)),
    'current_time':         str(datetime.now()),
    'data_age_hours':       float(data_age_hours),
    'target_window':        'Next 24 hours',
    'rain_probability':     float(rain_probability),
    'predicted_amount_mm':  float(predicted_amount),
    'final_prediction_mm':  float(final_prediction),
    'category':             category,
    'season':               current_season,
    'irrigation_advice':    irrigation_advice
}

print(f"\nPrediction complete")



MAKING PREDICTION
Feature order loaded from model: 71 features
First 4: ['hour', 'month', 'day_of_year', 'day_of_week']

Prediction time:  2026-03-07 11:00:00
Target window:    Next 24 hours
Expected features: 71
All 71 features present
Feature set compatible with model

Running Stage 1 (Classification)...
   P(rain >= 1.0mm) = 0.627
   Classification: RAIN

Running Stage 2 (Regression)...
   Predicted amount: 4.42 mm

FINAL PREDICTION

   Next 24-hour rainfall: 2.77 mm

   Breakdown:
      Rain probability:    62.7%
      Predicted amount:    4.42 mm
      Combined (P x amt):  2.77 mm

   Observation time: 2026-03-07 11:00
   Data age:         0.1 hours

INTERPRETATION

   Category:   Light rain
   For CWR:    Reduce irrigation by 30-50%
   Season:     Dry Season (Mar, Aug, Sep)

Prediction complete


## Cleaned complete cell

In [12]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import requests
import joblib
import warnings
warnings.filterwarnings('ignore')

# ── Configuration ─────────────────────────────────────────────────────────────
COLOMBO_LAT = 6.9271
COLOMBO_LON  = 79.8612

# ── Load models ───────────────────────────────────────────────────────────────
stage1_model = joblib.load('rainfall/stage1_rain_classifier.pkl')
stage2_model = joblib.load('rainfall/stage2_amount_regressor.pkl')
FEATURE_NAMES = stage1_model.get_booster().feature_names

# ── Fetch weather data ────────────────────────────────────────────────────────
current_time = datetime.now()
current_date = current_time.date()
hourly_vars  = ["temperature_2m", "relative_humidity_2m", "dew_point_2m",
                "precipitation", "surface_pressure", "cloud_cover",
                "wind_speed_10m", "wind_direction_10m"]

def fetch_df(url, params):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()['hourly']
    return pd.DataFrame({'datetime': pd.to_datetime(data['time']),
                         **{v: data[v] for v in hourly_vars}})

df_hist = fetch_df("https://archive-api.open-meteo.com/v1/archive", {
    "latitude": COLOMBO_LAT, "longitude": COLOMBO_LON,
    "start_date": str(current_date - timedelta(days=20)),
    "end_date":   str(current_date - timedelta(days=4)),
    "hourly": ",".join(hourly_vars), "timezone": "Asia/Colombo"
})

df_recent = fetch_df("https://api.open-meteo.com/v1/forecast", {
    "latitude": COLOMBO_LAT, "longitude": COLOMBO_LON,
    "hourly": ",".join(hourly_vars),
    "past_days": 4, "forecast_days": 1, "timezone": "Asia/Colombo"
})
df_recent = df_recent[df_recent['datetime'] <= current_time]

df = (pd.concat([df_hist, df_recent], ignore_index=True)
        .drop_duplicates(subset=['datetime'], keep='last')
        .sort_values('datetime').reset_index(drop=True))

# ── Feature engineering ───────────────────────────────────────────────────────
df['hour']        = df['datetime'].dt.hour
df['month']       = df['datetime'].dt.month
df['year']        = df['datetime'].dt.year
df['day_of_year'] = df['datetime'].dt.dayofyear
df['day_of_week'] = df['datetime'].dt.dayofweek
df['date_dt']     = df['datetime'].dt.normalize()

W, MP = 24, 18

df['precip_24h_sum']              = df['precipitation'].rolling(W, min_periods=MP).sum()
df['precip_24h_max']              = df['precipitation'].rolling(W, min_periods=MP).max()
df['precip_24h_mean']             = df['precipitation'].rolling(W, min_periods=MP).mean()
df['precip_24h_std']              = df['precipitation'].rolling(W, min_periods=MP).std()
df['precip_24h_rainy_hours']      = df['precipitation'].rolling(W, min_periods=MP).apply(lambda x: (x > 0.1).sum(), raw=True)

df['temp_24h_mean']  = df['temperature_2m'].rolling(W, min_periods=MP).mean()
df['temp_24h_max']   = df['temperature_2m'].rolling(W, min_periods=MP).max()
df['temp_24h_min']   = df['temperature_2m'].rolling(W, min_periods=MP).min()
df['temp_24h_range'] = df['temp_24h_max'] - df['temp_24h_min']
df['temp_24h_std']   = df['temperature_2m'].rolling(W, min_periods=MP).std()
df['temp_24h_trend'] = df['temperature_2m'] - df['temperature_2m'].shift(W)

df['humidity_24h_mean']           = df['relative_humidity_2m'].rolling(W, min_periods=MP).mean()
df['humidity_24h_max']            = df['relative_humidity_2m'].rolling(W, min_periods=MP).max()
df['humidity_24h_min']            = df['relative_humidity_2m'].rolling(W, min_periods=MP).min()
df['humidity_24h_range']          = df['humidity_24h_max'] - df['humidity_24h_min']
df['humidity_24h_hours_above_80'] = df['relative_humidity_2m'].rolling(W, min_periods=MP).apply(lambda x: (x > 80).sum(), raw=True)

df['pressure_24h_mean']  = df['surface_pressure'].rolling(W, min_periods=MP).mean()
df['pressure_24h_min']   = df['surface_pressure'].rolling(W, min_periods=MP).min()
df['pressure_24h_max']   = df['surface_pressure'].rolling(W, min_periods=MP).max()
df['pressure_24h_std']   = df['surface_pressure'].rolling(W, min_periods=MP).std()
df['pressure_24h_trend'] = df['surface_pressure'] - df['surface_pressure'].shift(W)

df['wind_24h_mean'] = df['wind_speed_10m'].rolling(W, min_periods=MP).mean()
df['wind_24h_max']  = df['wind_speed_10m'].rolling(W, min_periods=MP).max()
df['wind_24h_std']  = df['wind_speed_10m'].rolling(W, min_periods=MP).std()

df['cloud_24h_mean']              = df['cloud_cover'].rolling(W, min_periods=MP).mean()
df['cloud_24h_max']               = df['cloud_cover'].rolling(W, min_periods=MP).max()
df['cloud_24h_hours_above_90']    = df['cloud_cover'].rolling(W, min_periods=MP).apply(lambda x: (x > 90).sum(), raw=True)

df['dewpoint_24h_mean'] = df['dew_point_2m'].rolling(W, min_periods=MP).mean()

df['temp_current']                   = df['temperature_2m']
df['humidity_current']               = df['relative_humidity_2m']
df['pressure_current']               = df['surface_pressure']
df['dewpoint_current']               = df['dew_point_2m']
df['wind_speed_current']             = df['wind_speed_10m']
df['wind_direction_current']         = df['wind_direction_10m']
df['cloud_cover_current']            = df['cloud_cover']
df['dew_point_depression_current']   = df['temperature_2m'] - df['dew_point_2m']
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

daily_agg = df.groupby('date_dt').agg(
    daily_rainfall_total=('precipitation', 'sum'),
    daily_temp_mean=('temperature_2m', 'mean'),
    daily_pressure_mean=('surface_pressure', 'mean'),
    daily_humidity_mean=('relative_humidity_2m', 'mean'),
).reset_index().sort_values('date_dt').reset_index(drop=True)

for lag in [1, 2, 3, 7, 14]:
    daily_agg[f'rainfall_lag_{lag}d'] = daily_agg['daily_rainfall_total'].shift(lag)
for lag in [1, 3, 7]:
    daily_agg[f'temp_lag_{lag}d'] = daily_agg['daily_temp_mean'].shift(lag)
for lag in [1, 3]:
    daily_agg[f'pressure_lag_{lag}d'] = daily_agg['daily_pressure_mean'].shift(lag)
for lag in [1, 3]:
    daily_agg[f'humidity_lag_{lag}d'] = daily_agg['daily_humidity_mean'].shift(lag)

lag_cols = [c for c in daily_agg.columns if '_lag_' in c]
df = df.merge(daily_agg[['date_dt'] + lag_cols], on='date_dt', how='left')

df['month_sin']       = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos']       = np.cos(2 * np.pi * df['month'] / 12)
df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365.25)
df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365.25)

df['season_SW_Monsoon']  = df['month'].isin([5, 6, 7]).astype(int)
df['season_NE_Monsoon']  = df['month'].isin([12, 1, 2]).astype(int)
df['season_Inter_Heavy'] = df['month'].isin([4, 10, 11]).astype(int)
df['season_Inter_Light'] = (~df['month'].isin([5,6,7,12,1,2,4,10,11])).astype(int)

df['days_into_sw_monsoon'] = df.apply(
    lambda r: (r['datetime'] - pd.Timestamp(year=r['year'], month=5, day=1)).days
              if r['month'] in [5, 6, 7] else 0, axis=1)
df['days_into_ne_monsoon'] = df.apply(
    lambda r: (r['datetime'] - pd.Timestamp(
                   year=r['year'] if r['month'] == 12 else r['year'] - 1, month=12, day=1)).days
              if r['month'] in [12, 1, 2] else 0, axis=1)

df['pressure_tendency_3h']    = (df['surface_pressure'] - df['surface_pressure'].shift(3)) / 3
df['pressure_tendency_6h']    = (df['surface_pressure'] - df['surface_pressure'].shift(6)) / 6
df['wind_from_southwest']     = ((df['wind_direction_10m'] >= 180) & (df['wind_direction_10m'] <= 270)).astype(int)
df['wind_from_northeast']     = ((df['wind_direction_10m'] >= 0)   & (df['wind_direction_10m'] <= 90)).astype(int)
raw_change = np.abs(df['wind_direction_10m'] - df['wind_direction_10m'].shift(6))
df['wind_direction_change_6h'] = raw_change.apply(lambda x: min(x, 360 - x) if pd.notna(x) else x)
df['instability_index']       = (df['temp_24h_range'] * (1 - df['dew_point_depression_current'] / 20)).clip(lower=0)
df['moisture_flux']           = df['humidity_24h_mean'] * df['wind_24h_mean']

# ── Build prediction vector ───────────────────────────────────────────────────
latest = df.iloc[-1:]
X_pred = pd.DataFrame(index=[0], columns=FEATURE_NAMES, dtype=float)
for feat in FEATURE_NAMES:
    X_pred[feat] = latest[feat].values[0] if feat in latest.columns else 0.0
X_pred = X_pred.fillna(0).astype(float)

# ── Predict ───────────────────────────────────────────────────────────────────
rain_probability = stage1_model.predict_proba(X_pred)[0, 1]
predicted_amount = max(0, stage2_model.predict(X_pred)[0])
final_prediction = max(0, rain_probability * predicted_amount)

print(f"{final_prediction:.2f}")


2.77
